In [1]:
# Install dependecncies
%pip install anthropic python-dotenv

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 15.2 MB/s  0:00:007.1 MB/s eta 0:00:01
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [anthropic]━ 5/6 [anthropic]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

In [46]:
# Create an API Client

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-5-5"

In [47]:
# Make a request
def add_user_message(messages, text):
    user_message = { "role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = { "role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None, output_config=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    if system:
        params["system"] = system
    if output_config:
            params["output_config"] = output_config


    message = client.messages.create(**params)
    return "".join(
        block.text
        for block in message.content
        if block.type == "text"
    )

system = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

In [22]:
# Starting a list
messages = []

add_user_message(messages, "How much is 3 + 2 * 7")

answer = chat(messages)

add_assistant_message(messages, answer)

add_user_message(messages, "How much is 3 + 2 * 8")

answer_with_prompt = chat(messages, system)

add_assistant_message(messages, answer_with_prompt)

In [26]:
messages = []
add_user_message(messages, "Describe the best possible basketball game in 2 sentences")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

The best possible basketball game is a seven-game NBA Finals showdown between two evenly matched rivals, decided by a buzzer-beating shot in the final seconds of Game 7 after a back-and-forth battle that featured record-breaking performances and multiple lead changes. Fans on both sides would leave breathless, with every possession mattering and both teams giving their absolute maximum effort until the final horn.

In [49]:
import json
messages = []

add_user_message(messages, "Give me short evnet bridge rule in JSON format")
# add_assistant_message(messages, "```json")
text = chat(messages, output_config={
    "format": {
        "type": "json_schema",
        "schema": {
            "type": "object",
            "properties": {
                "key": { "type": "string" },
                "value": { "type": "string" }
            },
            "required": ["key", "value"],
            "additionalProperties": False  
        },
    }
})
json.loads(text)

{'key': 'EventBridge rule',
 'value': '{"source": ["aws.ec2"], "detail-type": ["EC2 Instance State-change Notification"], "detail": {"state": ["running"]}}'}